# Thesis: Reclaimed Timber in Deep Generative Design

**Notebook:** c25_26_27_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra     
**Last Updated:** 2026-02-27

## Cost and ILP Optimization Workflow

Generate a cost matrix for the geometry using timber datasets, then use ILP to find the best assignments.

**Inputs:** CSV timber dataset, Digital geometry  
**Outputs:** Best match for each structural element

# IMPORTING

In [ ]:
import config
import json
import pandas as pd

stock_path = config.TIMBER_STOCK_PATH / 'complete_timber.csv'
json_path = config.DATA_IO_PATH / "search_space.json"

with open(json_path, "r") as f:
    optimizer_search_space = json.load(f)

print(f"Search space loaded. The optimizer can control {len(optimizer_search_space)} parameters.")

# Try common combinations
read_attempts = [
    {"sep": ",", "encoding": "utf-8"},
    {"sep": ";", "encoding": "utf-8"},
    {"sep": ",", "encoding": "latin1"},
    {"sep": ";", "encoding": "latin1"},
]

df_input_stock = None
for opts in read_attempts:
    try:
        df_try = pd.read_csv(stock_path, **opts)  # type: ignore
        # Valid if we get more than 1 column
        if df_try.shape[1] > 1:
            df_input_stock = df_try
            print(f"Loaded with sep='{opts['sep']}' and encoding='{opts['encoding']}'")
            break
    except Exception:
        pass

if df_input_stock is None:
    raise ValueError("Could not parse CSV with tested delimiter/encoding combinations.")

# Clean column names
df_input_stock.columns = df_input_stock.columns.str.strip()

print("Detected columns:", df_input_stock.columns.tolist())
print(f"\nDataset contains {df_input_stock.shape[0]} elements\n")
display(df_input_stock.head())

# GEOMETRY

In [ ]:
import importlib
import matplotlib.pyplot as plt
import config
import c24_stage_geometry as stage_geometry

importlib.reload(stage_geometry)

geometry_out = stage_geometry.run_random_geometry_stage(
    json_path=json_path if "json_path" in globals() else None,
    optimizer_search_space=optimizer_search_space if "optimizer_search_space" in globals() else None,
    sample_id=0,
)

my_random_design = geometry_out["my_random_design"]
vertices_list = geometry_out["vertices_list"]
df_vertices = geometry_out["df_vertices"]
df_edges = geometry_out["df_edges"]
df_geometry_overview = geometry_out["df_geometry_overview"]

print(f"Geometry: {len(df_vertices)} nodes, {len(df_edges)} members")
print(
    f"Length range [m]: {df_geometry_overview['length_m'].min():.3f}"
    f" - {df_geometry_overview['length_m'].max():.3f}"
)
display(df_geometry_overview[["edge_id", "V1", "V2", "length_m"]].head(5))

# saving the df_vertices and df_edges for later use
df_vertices.to_csv(config.DATA_IO_PATH / "df_vertices.csv", index=False)
df_edges.to_csv(config.DATA_IO_PATH / "df_edges.csv", index=False)

fig, ax = stage_geometry.plot_geometry_preview(
    df_vertices=df_vertices,
    df_edges=df_edges,
    figsize=(8, 7),
)
plt.show()

# FEASIBILITY, COST, MATCHING, GNN AND FITNESS

In [ ]:
import importlib
import json
import pandas as pd
import config
from workflows import c26_stage_cost_matrix as stage_cost
from workflows import c27_stage_milp as stage_milp

importlib.reload(stage_cost)
importlib.reload(stage_milp)

# =============================================================================
# Feasibility Check: Filter cost matrix before MILP
# =============================================================================
# Uses the c25_stage_feasibility module to validate slot-stock combinations
# based on: 1) Length constraints, 2) EC5 structural capacity with buckling

import numpy as np
from workflows import c25_stage_feasibility as stage_feas

# =============================================================================
# EXTRACT GEOMETRY FROM PREVIOUS CELLS
# =============================================================================

# Parse geometry from df_vertices and df_edges (generated in GEOMETRY cell)
verts = df_vertices[df_vertices['sample_id'] == 0].copy()
verts['v_idx'] = verts['vertex_index'].str.replace('v', '').astype(int)
verts = verts.sort_values('v_idx').reset_index(drop=True)
node_positions = verts[['x', 'y', 'z']].values   # [39, 3] metres

support_nodes = verts[verts['attribute'] == 'support']['v_idx'].tolist()
load_nodes    = verts[verts['attribute'] == 'load']['v_idx'].tolist()

print(f"Geometry loaded:")
print(f"  Nodes: {len(verts)}, Members: {len(df_edges)}")
print(f"  Support nodes: {support_nodes}")
print(f"  Load nodes: {load_nodes}")
print()

# =============================================================================
# RUN FEASIBILITY FILTER
# =============================================================================

# Build feasibility mask for current geometry
df_slots, feasibility_mask, member_forces, stats = stage_feas.build_cost_filter(
    node_positions=node_positions,
    edges_df=df_edges,
    stock_df=df_input_stock,
    support_nodes=support_nodes,
    load_nodes=load_nodes,
)

# =============================================================================
# DISPLAY SUMMARY
# =============================================================================

print("\n" + "="*70)
print("FEASIBILITY FILTER SUMMARY")
print("="*70)
for k, v in stats.items():
    if isinstance(v, list):
        print(f"  {k:<35} {v}")
    elif isinstance(v, float):
        print(f"  {k:<35} {v:.4f}")
    else:
        print(f"  {k:<35} {v}")

print()
print(f"Member forces:")
print(f"  Tension     (N > 0): {(member_forces >  1.0).sum()} members")
print(f"  Compression (N < 0): {(member_forces < -1.0).sum()} members")
print(f"  Near-zero:           {(np.abs(member_forces) <= 1.0).sum()} members")
print(f"  Max tension:         {member_forces.max()/1000:.2f} kN")
print(f"  Max compression:     {member_forces.min()/1000:.2f} kN")

print("\n✓ Feasibility mask ready for cost matrix filtering")

In [ ]:

# -------------------------------
# STEP 1: COST MATRIX STAGE
# -------------------------------
cost_matrix, stock_prepared, logs = stage_cost.build_cost_matrix(
    df_slots=df_slots,
    df_input_stock=df_input_stock,
    feasibility_mask=feasibility_mask,
)

# save cost matrix for later analysis
cost_matrix_df = pd.DataFrame(cost_matrix)
cost_matrix_df.to_csv(config.EXPORT_PATH / "c26_cost_matrix.csv")

# -------------------------------
# STEP 2: MILP STAGE
# -------------------------------
print("Starting MILP optimizer...")
milp_out = stage_milp.run_milp_stage(
    cost_matrix=cost_matrix,
    enriched_stock=stock_prepared,
    df_slots=df_slots,
    reclaimed_marker="RS",
    new_marker="NS",
    new_stock_max_uses=100,
    solver_msg=False,
    raise_on_infeasible_slots=True,
)

status = milp_out["status"]
df_results = milp_out["df_results"]
total_cost = milp_out["total_cost"]
milp_summary = milp_out["summary"]

df_results.to_csv(config.EXPORT_PATH / "c27_milp_results.csv", index=False)

print(
    f"MILP setup: {milp_summary['reclaimed_items']} reclaimed + "
    f"{milp_summary['new_items']} new stock items for {milp_summary['slots']} slots"
)
print(f"MILP status: {status}")
print(f"Total assignment cost: {total_cost:.2f}")
if len(df_results) > 0:
    display(df_results.head(10))

In [ ]:
import c28_stage_GNN as stage_gnn

gnn_out = stage_gnn.run_gnn_stage(
    node_positions  = node_positions,
    milp_assignment = milp_out["milp_assignment"],  # from c27_stage_milp_v2
    df_input_stock  = df_input_stock,
    model_bundle    = model_bundle,
)
feasibility_score  = gnn_out["feasibility_score"]
structural_penalty = gnn_out["structural_penalty"]

print(f"\nGNN Feasibility Results:")
print(f"  Feasibility score:  {feasibility_score:.4f}  (1.0 = all members predicted safe)")
print(f"  Unsafe members:     {len(unsafe_member_ids)} / {stage_gnn.NUM_EDGES_PHYSICAL}")
print(f"  Structural penalty: {structural_penalty:.4f}  (w_structural=0.3)")
if unsafe_member_ids:
    print(f"  Unsafe member IDs:  {unsafe_member_ids[:20]}{'...' if len(unsafe_member_ids) > 20 else ''}")


In [ ]:
# -------------------------------
# STEP 3.5: FITNESS NORMALIZATION BOUNDS
# -------------------------------
from workflows.c29_stage_fitness_score import run_fitness_stage
from workflows.c29_stage_normalization_bounds import run_normalization_bounds_stage

bounds_out = run_normalization_bounds_stage(
    cost_matrix=cost_matrix,
    df_logs=logs,
    enriched_stock=stock_prepared,
    df_slots=df_slots,
    reclaimed_marker="RS",
    new_marker="NS",
    new_stock_max_uses=100,   # match your c27 call
    solver_msg=False,
    print_summary=True,
)

normalization_constants = bounds_out["normalization_constants"]
print(normalization_constants)

# -------------------------------
# STEP 4: FITNESS STAGE
# -------------------------------
fitness_weights = {
    "omega_1": 1.0,
    "omega_2": 1.0,
    "omega_3": 1.0,
    "omega_4": 1.0,   # structural penalty weight
}

fitness_out = run_fitness_stage(
    df_results=df_results,
    enriched_stock=stock_prepared,
    df_slots=df_slots,
    total_cost=total_cost,
    weight_config=fitness_weights,
    normalization_constants=normalization_constants,
    structural_infeasibility= 1.0 - feasibility_score,   # convert to penalty (0 = feasible, 1 = infeasible)
    derive_normalization_constants=False,
    run_sanity_checks=True,
    print_breakdown=True,
)

fitness_result = fitness_out["fitness_result"]
normalization_constants = fitness_out["normalization_constants"]

print("Fitness summary:")
print(f"  objective: {fitness_result.get('objective', 'n/a')}")
print(f"  feasible: {fitness_result.get('is_feasible', 'n/a')}")
print(f"  fitness: {fitness_result.get('fitness', 'n/a')}\n")

# Notebook-owned fitness exports
fitness_json_path = config.EXPORT_PATH / "c28_fitness_result.json"
fitness_csv_path = config.EXPORT_PATH / "c28_fitness_result.csv"

fitness_json_path.parent.mkdir(parents=True, exist_ok=True)

# Convert numpy scalars to native Python values for stable JSON serialization.
def _to_builtin(value):
    return value.item() if hasattr(value, "item") else value

fitness_export.pop("weights", None)   # remove the tuple from evaluate_milp_solution
fitness_export["weights"] = {         # replace with your explicit named dict
    "omega_1": float(fitness_weights["omega_1"]),
    "omega_2": float(fitness_weights["omega_2"]),
    "omega_3": float(fitness_weights["omega_3"]),
    "omega_4": float(fitness_weights["omega_4"]),
}
fitness_export["normalization_constants"] = {
    "C_max": float(normalization_constants["C_max"]),
    "R_max": float(normalization_constants["R_max"]),
    "W_max": float(normalization_constants["W_max"]),
}

with open(fitness_json_path, "w", encoding="utf-8") as f:
    json.dump(fitness_export, f, indent=2)

fitness_row = {
    **{key: _to_builtin(value) for key, value in fitness_result.items()},
    **fitness_weights,
    "C_max": float(normalization_constants["C_max"]),
    "R_max": float(normalization_constants["R_max"]),
    "W_max": float(normalization_constants["W_max"]),
}
pd.DataFrame([fitness_row]).to_csv(fitness_csv_path, index=False)

print(f"Exported fitness JSON: {fitness_json_path}")
print(f"Exported fitness CSV: {fitness_csv_path}")

# EXPORT

Exports structural parameters (vertices + edges with assigned timber) for Grasshopper reconstruction.

In [ ]:
# -------------------------------
# Step 3: GNN FEASIBILITY CHECK
# -------------------------------
import importlib
import numpy as np
import torch
import config
import c21_surrogate_model_v4
from workflows import c28_stage_GNN as stage_gnn

importlib.reload(c21_surrogate_model_v4)
importlib.reload(stage_gnn)

# =============================================================================
# LOAD MODEL (once per session — skip expensive disk read on re-runs)
# =============================================================================
if "model_bundle" not in globals():
    ARTIFACT_STEM = "ID20260510_224228_LR3e-04_EP150_BS32_FA0.50_ROC0.874"
    _d = config.SM_EXPORT_PATH / ARTIFACT_STEM
    model_bundle = stage_gnn.load_gnn_model(
        ckpt_path             = _d / f"{ARTIFACT_STEM}.pth",
        norm_stats_path       = _d / f"{ARTIFACT_STEM}_norm_stats.pt",
        edge_index_path       = _d / f"{ARTIFACT_STEM}_edge_index.json",
        inference_config_path = _d / f"{ARTIFACT_STEM}_inference_config.json",
        device                = "cuda" if torch.cuda.is_available() else "cpu",
    )

# =============================================================================
# STOCK FEATURES FOR GNN  (mm / N·mm⁻² → SI)
# =============================================================================
stock_gnn = df_input_stock.copy()
b_m  = stock_gnn["Width"].values         * 1e-3
h_m  = stock_gnn["Depth"].values         * 1e-3
L_m  = stock_gnn["Length"].values        * 1e-3
E_pa = stock_gnn["E_modulus_eff"].values * 1e6
A    = b_m * h_m
a, c = np.minimum(b_m, h_m), np.maximum(b_m, h_m)   # short/long side for torsion

stock_gnn["Area"]   = A
stock_gnn["Length"] = L_m
stock_gnn["E"]      = E_pa
stock_gnn["Iy"]     = b_m * h_m**3 / 12
stock_gnn["Iz"]     = h_m * b_m**3 / 12
stock_gnn["J"]      = a**3 * c / 3 * (1 - 0.63 * a / c)
stock_gnn["EA/L"]   = E_pa * A / L_m

# =============================================================================
# milp_assignment — [120] stock row indices ordered by edge number
# =============================================================================
stock_id_to_idx = {mid: i for i, mid in enumerate(df_input_stock["Member_ID"])}
milp_assignment = (
    df_results
    .assign(edge_num=df_results["edge_id"].str[1:].astype(int))
    .sort_values("edge_num")["assigned_timber"]
    .map(stock_id_to_idx)
    .to_numpy()
)

# =============================================================================
# GNN FORWARD PASS
# =============================================================================
feasibility_score, unsafe_member_ids, preds_physical = stage_gnn.gnn_feasibility(
    node_positions  = node_positions,
    milp_assignment = milp_assignment,
    stock_df        = stock_gnn,
    model_bundle    = model_bundle,
)

structural_penalty = stage_gnn.structural_fitness_penalty(feasibility_score, unsafe_member_ids)

print(f"GNN: score={feasibility_score:.4f}  unsafe={len(unsafe_member_ids)}/{stage_gnn.NUM_EDGES_PHYSICAL}  penalty={structural_penalty:.4f}")
if unsafe_member_ids:
    print(f"  Unsafe IDs: {unsafe_member_ids[:20]}{'...' if len(unsafe_member_ids) > 20 else ''}")

# LITERATURE

### COST MATRIX
We now use one universal assignment formula for both reclaimed and new timber in the same cost matrix:

$$C_{i,j} = E_{embodied,i} + E_{prep,i} + E_{trans,i,j} + E_{waste,i,j} + E_{saw,i,j}$$

With:
- $E_{embodied,i} = V_{stock,i} \cdot M_{material,i}$ (from `ECC` in stock data)
- $E_{prep,i} = V_{stock,i} \cdot E_{prep}$
- $E_{trans,i,j} = \left(\frac{(V_{req}+V_{over})\cdot\rho}{1000}\right)\cdot D\cdot F_{trans}$
- $E_{waste,i,j} = V_{waste} \cdot E_{EoL}$
- $E_{saw,i,j} = P_{cut}$ when cutting is required, else 0

Interpretation by timber type:
- New timber: typically high `ECC`, usually low/zero preparation factor
- Reclaimed timber: `ECC` near 0, non-zero preparation factor

### MATCHING ALGORITHM / MILP
The assignment problem in this notebook is solved as a mixed-integer linear program (MILP). Let:

- $I$ be the set of required structural slots,
- $J$ be the set of inventory stock elements,
- $F \subseteq I \times J$ be the set of physically feasible slot-stock combinations,
- $R \subseteq J$ be the subset of reclaimed timber,
- $N \subseteq J$ be the subset of new timber,
- $c_{ij}$ be the cost of assigning stock element $j$ to slot $i$,
- $x_{ij} \in \{0,1\}$ be the decision variable, where $x_{ij}=1$ means that stock element $j$ is assigned to slot $i$.

$$
\min_{x} \sum_{(i,j) \in F} c_{ij} x_{ij}
$$

subject to

$$
\sum_{j:(i,j)\in F} x_{ij} = 1 \qquad \forall i \in I
$$

$$
\sum_{i:(i,j)\in F} x_{ij} \le 1 \qquad \forall j \in R
$$

$$
\sum_{i:(i,j)\in F} x_{ij} \le |I| \qquad \forall j \in N
$$

$$
x_{ij} = 0 \qquad \forall (i,j) \notin F
$$

In words: every structural member must receive exactly one feasible timber element, reclaimed timber can be used at most once, and new timber can be reused when physically feasible.

### MULTI-OBJECTIVE FITNESS FUNCTION

The fitness function balances three competing objectives:

$$F(\mathbf{x}) = \omega_1 \left( \frac{f_{inner}^*(\mathbf{x})}{\mathcal{C}_{max}} \right) - \omega_2 \left( \frac{\mathcal{R}(\mathbf{x}, \mathbf{y}^*)}{\mathcal{R}_{max}} \right) + \omega_3 \left( \frac{\mathcal{W}(\mathbf{x}, \mathbf{y}^*)}{\mathcal{W}_{max}} \right)$$

- **$f_{inner}^*$**: MILP cost (kg CO2e) — penalizes virgin material and waste
- **$\mathcal{R}$**: Reuse rate (%) — reward for using reclaimed timber (subtracted, so higher reuse = better)
- **$\mathcal{W}$**: Total waste (m³) — penalizes inefficient cutting
- **$\omega_i$**: Weight coefficients to tune priorities

All metrics are normalized to [0, 1] using precomputed dataset-driven extremes ($\mathcal{C}_{max}$, $\mathcal{R}_{max}$, $\mathcal{W}_{max}$).

**Design philosophy**: The MILP cost matrix already accounts for all physical and geometric infeasibilities (by returning ∞ for impossible assignments). This means the upper-level fitness function does NOT need a conditional fallback loop. Instead, we can directly minimize a weighted multi-objective sum.

**Sign convention**:
- Positive coefficient on cost: higher MILP cost → higher (worse) fitness
- Negative coefficient on reuse: higher reuse rate → lower (better) fitness ✓
- Positive coefficient on waste: higher waste → higher (worse) fitness

**Typical design trade-offs**:
- High ω₁ (cost weight): prioritize LCA minimization across all materials
- High ω₂ (reuse weight): prioritize reclaimed material recovery
- High ω₃ (waste weight): prioritize cutting efficiency